# EBRAINS BrainScaleS-2 toy ANN2SNN manual hardware run

Use the `EBRAINS-experimental` kernel and run cells from top to bottom. This clean launcher reuses the accepted Yin-Yang checkpoint and explicit calibration files, checks the hardware service with a bounded subprocess, and then runs deadline-margin calibration, a same-run smoke gate, and the full evaluation.

Experiment logic remains in `scripts/evaluation/brainscales2_toy_hil.py`. No API token belongs in this notebook.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import os
import signal
import subprocess
import sys
import time

start = Path.cwd().resolve()
repo_root = next(
    (path for path in (start, *start.parents)
     if (path / 'scripts/evaluation/brainscales2_toy_hil.py').is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError('Could not locate /mnt/user/shared/AnalogAttention')
os.chdir(repo_root)
CLI = repo_root / 'scripts/evaluation/brainscales2_toy_hil.py'

# Run All executes the physical Yin-Yang acceptance path. A failed cell stops
# later cells, and full evaluation additionally requires this run's smoke gate.
RUN_SERVICE_PREFLIGHT = True
RUN_HAGEN_PROBE = False  # The accepted seed-0 checkpoint already selected shift 2.
RUN_MARGIN_CALIBRATION = True
RUN_HARDWARE_SMOKE = True
RUN_YINYANG_FULL = True

SOURCE_RUN = repo_root / 'artifacts/brainscales2-toy/20260901T095827Z'
CHECKPOINT_DIR = SOURCE_RUN / 'checkpoint'
HAGEN_CALIBRATION_PATH = SOURCE_RUN / 'calibration/hagen_cocolist.pbin'
SPIKING_CALIBRATION_PATH = SOURCE_RUN / 'calibration/spiking_cocolist.pbin'
EXISTING_DEADLINE_MARGIN_PATH = None
HAGEN_HIDDEN_SHIFT = 2

TOY_ACTIVATION = 'relu'
RELU_BOUNDARY = 'implicit-lower-bound-host'
POOLING_DOMAIN = 'ttfs'
TEMPORAL_POOL_ESTIMATOR = 'analytic-corrected-max'
BASE_DEADLINE_S = 60.0e-6
MARGIN_DIAGNOSTIC_DEADLINE_S = 100.0e-6
MARGIN_MAX_S = 40.0e-6
MARGIN_STEP_S = 1.0e-6
MARGIN_TARGET_SAMPLE_MISS_RATE = 0.05
MARGIN_CONFIDENCE = 0.95
MARGIN_CALIBRATION_SAMPLES = 256
MARGIN_CALIBRATION_TRIALS = 8
MARGIN_BOOTSTRAP_ITERATIONS = 2000
SPIKING_THRESHOLD = 125
SPIKING_INPUT_FAN_IN = 4
POOL_SAMPLE_CHUNK_SIZE = 64
POOL_REPLICA_SAMPLE_BUDGET = 128
POOL_CALIBRATION_TRIAL_CHUNK_SIZE = 4
HAGEN_ROW_CHUNK_SIZE = 128
CONDITION_WORKER_MAX_ATTEMPTS = 3
CONDITION_WORKER_RETRY_BACKOFF_S = 20.0
CONDITION_WORKER_IDLE_TIMEOUT_S = 180.0
SMOKE_MAX_MULTI_SPIKE_RATE = 0.05

PREFLIGHT_TIMEOUT_S = 120
HAGEN_PROBE_TIMEOUT_S = 20 * 60
MARGIN_TIMEOUT_S = 2 * 60 * 60
SMOKE_TIMEOUT_S = 2 * 60 * 60
FULL_TIMEOUT_S = 12 * 60 * 60

run_label = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_manual')
artifact_root = repo_root / 'artifacts/brainscales2-toy' / run_label
artifact_root.mkdir(parents=True, exist_ok=True)

required = [
    CHECKPOINT_DIR / 'checkpoint.pt',
    CHECKPOINT_DIR / 'converted_checkpoint.pt',
    HAGEN_CALIBRATION_PATH,
    SPIKING_CALIBRATION_PATH,
]
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing required inputs: {missing}')

print('repository:', repo_root)
print('python:', sys.executable)
print('source run:', SOURCE_RUN)
print('new artifacts:', artifact_root)
print('Hagen calibration:', HAGEN_CALIBRATION_PATH)
print('spiking calibration:', SPIKING_CALIBRATION_PATH)

## Configure and check the hardware client

This cell uses the official demo helper from a writable `/tmp` checkout. It does not download or overwrite calibration files.

In [ ]:
demos_root = Path('/tmp/brainscales2-demos')
if not demos_root.is_dir():
    subprocess.run(
        [
            'git', 'clone', '--depth', '1', '--branch',
            'jupyter-notebooks-experimental',
            'https://github.com/electronicvisions/brainscales2-demos.git',
            str(demos_root),
        ],
        check=True,
    )
if str(demos_root) not in sys.path:
    sys.path.insert(0, str(demos_root))

from _static.common.helpers import setup_hardware_client

hardware_requested = any((
    RUN_SERVICE_PREFLIGHT,
    RUN_HAGEN_PROBE,
    RUN_MARGIN_CALIBRATION,
    RUN_HARDWARE_SMOKE,
    RUN_YINYANG_FULL,
))
HARDWARE_CLIENT_READY = False
if hardware_requested:
    started = time.monotonic()
    setup_hardware_client()
    HARDWARE_CLIENT_READY = True
    print(f'hardware client ready in {time.monotonic() - started:.1f} s')

## Bounded Hagen initialization preflight

The check runs in a disposable process, applies the explicit Hagen calibration, releases hardware immediately on success, and kills the process group after the configured timeout. Later stages are blocked when this check fails.

In [ ]:
def run_hagen_preflight(timeout_s):
    code = f'''
import hxtorch
print("initializing explicit Hagen calibration", flush=True)
try:
    hxtorch.init_hardware(hxtorch.CalibrationPath({str(HAGEN_CALIBRATION_PATH)!r}))
    print("Hagen initialization passed", flush=True)
    print("chip:", hxtorch.get_unique_identifier(), flush=True)
finally:
    try:
        hxtorch.release_hardware()
        print("hardware released", flush=True)
    except Exception as error:
        print("release failed:", type(error).__name__, error, flush=True)
'''
    started = time.monotonic()
    process = subprocess.Popen(
        [sys.executable, '-c', code],
        cwd=repo_root,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        start_new_session=True,
    )
    timed_out = False
    try:
        stdout, stderr = process.communicate(timeout=timeout_s)
    except subprocess.TimeoutExpired:
        timed_out = True
        os.killpg(process.pid, signal.SIGKILL)
        stdout, stderr = process.communicate()
    result = {
        'returncode': process.returncode,
        'timed_out': timed_out,
        'elapsed_s': time.monotonic() - started,
        'stdout': stdout,
        'stderr': stderr,
    }
    (artifact_root / 'service_preflight.json').write_text(
        json.dumps(result, indent=2), encoding='utf-8'
    )
    print(stdout)
    if stderr:
        print(stderr, file=sys.stderr)
    if timed_out:
        raise TimeoutError(f'Hagen initialization exceeded {timeout_s} seconds')
    if process.returncode != 0:
        raise RuntimeError(f'Hagen initialization failed with exit code {process.returncode}')
    return result

PREFLIGHT_OK = False
if RUN_SERVICE_PREFLIGHT:
    if not HARDWARE_CLIENT_READY:
        raise RuntimeError('Hardware client was not configured')
    preflight_result = run_hagen_preflight(PREFLIGHT_TIMEOUT_S)
    PREFLIGHT_OK = True
else:
    print('Service preflight disabled')

## CLI and artifact helpers

Every stage is a subprocess with live output and a process-group timeout. Stage status is written after each transition so a failed manual run remains diagnosable.

In [ ]:
pipeline_status = {}
status_path = artifact_root / 'manual_pipeline_status.json'

def write_pipeline_status():
    status_path.write_text(
        json.dumps(pipeline_status, indent=2, sort_keys=True, default=str),
        encoding='utf-8',
    )

def require_preflight():
    if RUN_SERVICE_PREFLIGHT and not PREFLIGHT_OK:
        raise RuntimeError('The same-run Hagen service preflight did not pass')

def run_cli_stage(name, timeout_s, *arguments):
    require_preflight()
    command = [sys.executable, str(CLI), *(str(value) for value in arguments)]
    print(f'\n=== {name} ===', flush=True)
    print(' '.join(command), flush=True)
    pipeline_status[name] = {'status': 'running', 'command': command}
    write_pipeline_status()
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=repo_root, start_new_session=True)
    try:
        returncode = process.wait(timeout=timeout_s)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
        pipeline_status[name] = {
            'status': 'failed',
            'error': f'timeout after {timeout_s} seconds',
            'elapsed_s': time.monotonic() - started,
        }
        write_pipeline_status()
        raise TimeoutError(pipeline_status[name]['error'])
    if returncode != 0:
        pipeline_status[name] = {
            'status': 'failed',
            'error': f'exit code {returncode}',
            'elapsed_s': time.monotonic() - started,
        }
        write_pipeline_status()
        raise subprocess.CalledProcessError(returncode, command)
    pipeline_status[name] = {
        'status': 'passed',
        'elapsed_s': time.monotonic() - started,
    }
    write_pipeline_status()

def calibration_args():
    return [
        '--hagen-calibration', HAGEN_CALIBRATION_PATH,
        '--spiking-calibration', SPIKING_CALIBRATION_PATH,
    ]

def base_hardware_args():
    return [
        '--activation', TOY_ACTIVATION,
        '--checkpoint', CHECKPOINT_DIR / 'checkpoint.pt',
        '--converted-checkpoint', CHECKPOINT_DIR / 'converted_checkpoint.pt',
        '--pwm-backend', 'hagen-hardware',
        '--pool-backend', 'hardware',
        '--pooling-domain', POOLING_DOMAIN,
        '--temporal-pool-estimator', TEMPORAL_POOL_ESTIMATOR,
        '--deadline-s', BASE_DEADLINE_S,
        '--hagen-hidden-shift', HAGEN_HIDDEN_SHIFT,
        '--relu-boundary', RELU_BOUNDARY,
        '--threshold', SPIKING_THRESHOLD,
        '--input-fan-in', SPIKING_INPUT_FAN_IN,
        '--pool-sample-chunk-size', POOL_SAMPLE_CHUNK_SIZE,
        '--pool-replica-sample-budget', POOL_REPLICA_SAMPLE_BUDGET,
        '--pool-calibration-trial-chunk-size', POOL_CALIBRATION_TRIAL_CHUNK_SIZE,
        '--hagen-row-chunk-size', HAGEN_ROW_CHUNK_SIZE,
        '--condition-worker-max-attempts', CONDITION_WORKER_MAX_ATTEMPTS,
        '--condition-worker-retry-backoff-s', CONDITION_WORKER_RETRY_BACKOFF_S,
        '--condition-worker-idle-timeout-s', CONDITION_WORKER_IDLE_TIMEOUT_S,
        *calibration_args(),
    ]

## Optional Hagen shift probe

The default reuses shift 2 from the accepted seed-0 checkpoint probe. Enable `RUN_HAGEN_PROBE` to acquire a new physical probe and use its selected shift in all later stages.

In [ ]:
if RUN_HAGEN_PROBE:
    probe_dir = artifact_root / 'hagen_probe'
    run_cli_stage(
        'hagen-probe', HAGEN_PROBE_TIMEOUT_S,
        '--phase', 'probe-hagen',
        '--task', 'yinyang',
        '--architecture', 'yy-30',
        '--activation', TOY_ACTIVATION,
        '--checkpoint', CHECKPOINT_DIR / 'checkpoint.pt',
        '--converted-checkpoint', CHECKPOINT_DIR / 'converted_checkpoint.pt',
        '--pwm-backend', 'hagen-hardware',
        '--hagen-hidden-shift', HAGEN_HIDDEN_SHIFT,
        '--relu-boundary', RELU_BOUNDARY,
        '--output-dir', probe_dir,
        *calibration_args(),
    )
    probe = json.loads((probe_dir / 'hagen_probe.json').read_text(encoding='utf-8'))
    HAGEN_HIDDEN_SHIFT = int(probe['hidden_shift_calibration']['selected']['shift'])
print('HAGEN_HIDDEN_SHIFT =', HAGEN_HIDDEN_SHIFT)

## Select the deadline margin

The calibration uses unlabeled inputs and writes the selected deadline extension into the new artifact directory.

In [ ]:
if RUN_MARGIN_CALIBRATION:
    margin_dir = artifact_root / 'deadline_margin'
    run_cli_stage(
        'deadline-margin-calibration', MARGIN_TIMEOUT_S,
        '--phase', 'calibrate-margin',
        '--task', 'yinyang',
        '--architecture', 'yy-30',
        '--margin-calibration-samples', MARGIN_CALIBRATION_SAMPLES,
        '--margin-calibration-trials', MARGIN_CALIBRATION_TRIALS,
        '--margin-diagnostic-deadline-s', MARGIN_DIAGNOSTIC_DEADLINE_S,
        '--margin-max-s', MARGIN_MAX_S,
        '--margin-step-s', MARGIN_STEP_S,
        '--margin-target-sample-miss-rate', MARGIN_TARGET_SAMPLE_MISS_RATE,
        '--margin-confidence', MARGIN_CONFIDENCE,
        '--margin-bootstrap-iterations', MARGIN_BOOTSTRAP_ITERATIONS,
        '--output-dir', margin_dir,
        *base_hardware_args(),
    )
    DEADLINE_MARGIN_PATH = margin_dir / 'deadline_margin.json'
elif EXISTING_DEADLINE_MARGIN_PATH is not None:
    DEADLINE_MARGIN_PATH = Path(EXISTING_DEADLINE_MARGIN_PATH).expanduser().resolve()
else:
    raise RuntimeError('Enable margin calibration or set EXISTING_DEADLINE_MARGIN_PATH')

margin_payload = json.loads(DEADLINE_MARGIN_PATH.read_text(encoding='utf-8'))
if not margin_payload.get('viable') or margin_payload.get('selected_margin_s') is None:
    raise RuntimeError(f'No viable deadline margin: {margin_payload.get("structural_floor")}')
print('selected margin [s]:', margin_payload['selected_margin_s'])
print('selected deadline [s]:', margin_payload['selected_deadline_s'])

def common_hardware_args():
    return [
        *base_hardware_args(),
        '--deadline-margin-json', DEADLINE_MARGIN_PATH,
        '--include-zero-margin-control',
    ]

## Same-run hardware smoke

The formal evaluation remains blocked unless this cell produces finite accuracy, at least one accepted spike, and an acceptable multi-spike rate for both M=1 and M=4.

In [ ]:
SMOKE_OK = False
if RUN_HARDWARE_SMOKE:
    smoke_dir = artifact_root / 'hardware_smoke'
    run_cli_stage(
        'hardware-smoke', SMOKE_TIMEOUT_S,
        '--phase', 'hardware-smoke',
        '--task', 'yinyang',
        '--architecture', 'yy-30',
        '--quick',
        '--output-dir', smoke_dir,
        *common_hardware_args(),
    )
    with (smoke_dir / 'metrics.csv').open(newline='', encoding='utf-8') as handle:
        smoke_rows = [row for row in csv.DictReader(handle) if row.get('pool_size')]
    observed_sizes = {int(row['pool_size']) for row in smoke_rows}
    if not {1, 4}.issubset(observed_sizes):
        raise RuntimeError(f'Smoke is missing M=1 or M=4: {observed_sizes}')
    for row in smoke_rows:
        if float(row['neuron_miss_rate']) >= 1.0:
            raise RuntimeError(f'All physical neurons missed: {row}')
        if float(row['multi_spike_rate']) > SMOKE_MAX_MULTI_SPIKE_RATE:
            raise RuntimeError(f'Multi-spike rate exceeds the smoke gate: {row}')
        if not row.get('accuracy'):
            raise RuntimeError(f'Missing smoke accuracy: {row}')
    SMOKE_OK = True
    print('same-run smoke passed with pool sizes:', sorted(observed_sizes))
else:
    print('Hardware smoke disabled')

## Full Yin-Yang evaluation

This stage evaluates M in {1, 2, 4, 8, 16} for both placement conditions and stops unless the same notebook run passed its smoke gate.

In [ ]:
if RUN_YINYANG_FULL:
    if not RUN_HARDWARE_SMOKE or not SMOKE_OK:
        raise RuntimeError('Full evaluation requires a passing same-run smoke gate')
    run_cli_stage(
        'yinyang-full', FULL_TIMEOUT_S,
        '--phase', 'hardware-eval',
        '--task', 'yinyang',
        '--architecture', 'yy-30',
        '--activation', TOY_ACTIVATION,
        '--pool-sizes', 1, 2, 4, 8, 16,
        '--output-dir', artifact_root / 'yinyang_full',
        *common_hardware_args(),
    )
else:
    print('Full Yin-Yang evaluation disabled')

## Inspect results

This final cell prints the stage status and the main metrics written by completed stages.

In [ ]:
print(json.dumps(pipeline_status, indent=2, default=str))
for name in ('hardware_smoke', 'yinyang_full'):
    metrics_path = artifact_root / name / 'metrics.csv'
    if metrics_path.is_file():
        print(f'\n{name}: {metrics_path}')
        with metrics_path.open(newline='', encoding='utf-8') as handle:
            rows = list(csv.DictReader(handle))
        for row in rows:
            if row.get('pool_size'):
                print({
                    key: row.get(key)
                    for key in (
                        'pool_size', 'placement', 'accuracy', 'accuracy_drop',
                        'neuron_miss_rate', 'all_miss_rate',
                        'oracle_miss_repair_accuracy', 'torch_readout_accuracy',
                    )
                })
print('artifacts:', artifact_root)